In [21]:
from torch.utils.data import Dataset,DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
import torch
from einops import rearrange

import torch.nn as nn
import pytorch_lightning as pl
import math
from utils.modules import *

/mnt/d/labaratory/labo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
diffusion_steps = 1000
epochs = 100
batch_size = 128

In [ ]:
class Data_setting(Dataset):
    def __init__(self, train):
        transform = transforms.Compose([transforms.ToTensor()])
        train_dataset = CIFAR10( "../dataset/mnist",
                                train = train,
                                download= True,
                                transform = transform
                                )
        self.dataset_len = len(train_dataset)
        self.depth = 3
        self.size = 32
        input_seq = ((train_dataset.data/255.0) * 2.0) -1.0
        self.input_seq = rearrange(input_seq, 'b h w c -> b c h w')
        
    def __len__(self):
        return self.dataset_len
    
    def __getitem__(self, item):
        return self.input_seq[item]
    
        

In [19]:
train_dataset = Data_setting(True)
val_dataset = Data_setting(False)

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle= True)
val_loader = DataLoader(val_dataset,
                          batch_size=batch_size,
                          shuffle= True)

In [22]:
class DiffusionModel(pl.LightningModule):
    def __init__(self, in_size, t_range, img_depth):
        super().__init__()
        self.beta_small = 1e-4
        self.beta_large = 0.02
        self.t_range = t_range
        self.in_size = in_size

        bilinear = True
        self.inc = DoubleConv(img_depth, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        factor = 2 if bilinear else 1
        self.down3 = Down(256, 512 // factor)
        self.up1 = Up(512, 256 // factor, bilinear)
        self.up2 = Up(256, 128 // factor, bilinear)
        self.up3 = Up(128, 64, bilinear)
        self.outc = OutConv(64, img_depth)
        self.sa1 = SAWrapper(256, 8)
        self.sa2 = SAWrapper(256, 4)
        self.sa3 = SAWrapper(128, 8)

    def pos_encoding(self, t, channels, embed_size):
        inv_freq = 1.0 / (
            10000
            ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc.view(-1, channels, 1, 1).repeat(1, 1, embed_size, embed_size)

    def forward(self, x, t):
        """
        Model is U-Net with added positional encodings and self-attention layers.
        """
        x1 = self.inc(x)
        x2 = self.down1(x1) + self.pos_encoding(t, 128, 16)
        x3 = self.down2(x2) + self.pos_encoding(t, 256, 8)
        x3 = self.sa1(x3)
        x4 = self.down3(x3) + self.pos_encoding(t, 256, 4)
        x4 = self.sa2(x4)
        x = self.up1(x4, x3) + self.pos_encoding(t, 128, 8)
        x = self.sa3(x)
        x = self.up2(x, x2) + self.pos_encoding(t, 64, 16)
        x = self.up3(x, x1) + self.pos_encoding(t, 64, 32)
        output = self.outc(x)
        return output

    def beta(self, t):
        return self.beta_small + (t / self.t_range) * (
            self.beta_large - self.beta_small
        )

    def alpha(self, t):
        return 1 - self.beta(t)

    def alpha_bar(self, t):
        return math.prod([self.alpha(j) for j in range(t)])

    def get_loss(self, batch, batch_idx):
        """
        Corresponds to Algorithm 1 from (Ho et al., 2020).
        """
        ts = torch.randint(0, self.t_range, [batch.shape[0]], device=self.device)
        noise_imgs = []
        epsilons = torch.randn(batch.shape, device=self.device)
        for i in range(len(ts)):
            a_hat = self.alpha_bar(ts[i])
            noise_imgs.append(
                (math.sqrt(a_hat) * batch[i]) + (math.sqrt(1 - a_hat) * epsilons[i])
            )
        noise_imgs = torch.stack(noise_imgs, dim=0)
        e_hat = self.forward(noise_imgs, ts.unsqueeze(-1).type(torch.float))
        loss = nn.functional.mse_loss(
            e_hat.reshape(-1, self.in_size), epsilons.reshape(-1, self.in_size)
        )
        return loss

    def denoise_sample(self, x, t):
        """
        Corresponds to the inner loop of Algorithm 2 from (Ho et al., 2020).
        """
        with torch.no_grad():
            if t > 1:
                z = torch.randn(x.shape)
            else:
                z = 0
            e_hat = self.forward(x, t.view(1, 1).repeat(x.shape[0], 1))
            pre_scale = 1 / math.sqrt(self.alpha(t))
            e_scale = (1 - self.alpha(t)) / math.sqrt(1 - self.alpha_bar(t))
            post_sigma = math.sqrt(self.beta(t)) * z
            x = pre_scale * (x - e_scale * e_hat) + post_sigma
            return x

    def training_step(self, batch, batch_idx):
        loss = self.get_loss(batch, batch_idx)
        self.log("train/loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        loss = self.get_loss(batch, batch_idx)
        self.log("val/loss", loss)
        return

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=2e-4)
        return optimizer

In [23]:
model = DiffusionModel(train_dataset.size * train_dataset.size,
                       diffusion_steps,
                       train_dataset.depth)

tb_logger= pl.loggers.TensorBoardLogger(
    '../logger/lightning_logs/diffusion/',
    name = 'DDPM_test'
)

trainer = pl.Trainer(
    max_epochs=epochs,
    log_every_n_steps= 10,
    logger= tb_logger
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [24]:
trainer.fit(model, train_loader, val_loader)

You are using a CUDA device ('NVIDIA GeForce RTX 4080') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ inc   │ DoubleConv │ 38.8 K │ train │     0 │
│ 1  │ down1 │ Down       │  295 K │ train │     0 │
│ 2  │ down2 │ Down       │  1.2 M │ train │     0 │
│ 3  │ down3 │ Down       │  2.4 M │ train │     0 │
│ 4  │ up1   │ Up         │  6.2 M │ train │     0 │
│ 5  │ up2   │ Up         │  1.5 M │ train │     0 │
│ 6  │ up3   │ Up         │  406 K │ train │     0 │
│ 7  │ outc  │ OutConv    │    195 │ train │     0 │
│ 8  │ sa1   │ SAWrapper  │  395 K │ train │     0 │
│ 9  │ sa2   │ SAWrapper  │  395 K │ train │     0 │
│ 10 │ sa3   │ SAWrapper  │ 99.6 K │ train │     0 │
└────┴───────┴────────────┴────────┴───────┴───────┘

Trainable params: 12.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 12.9 M                                                                                               
Total estimated model params size (MB): 51                                                                         
Modules in train mode: 141                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/mnt/d/labaratory/labo/lib/python3.12/site-packages/rich/live.py:256: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

/mnt/d/labaratory/labo/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:485: 
Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for 
val/test dataloaders.

/mnt/d/labaratory/labo/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


KeyboardInterrupt

